In [1]:
!pip install -q faster-whisper transformers==4.52.4 sentencepiece accelerate moviepy jiwer rouge-score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.5 MB/s eta 0:00:00


Creating Project Folders

In [2]:
import os

folders = [
    "videos",
    "audios",
    "transcripts",
    "subtitles",
    "summaries"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully")

Folders created successfully


**SRT Timestamp Function**

In [3]:
def format_time(seconds):

    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)

    return f"{hrs:02}:{mins:02}:{secs:02},{millis:03}"

In [4]:
print(format_time(125.678))

00:02:05,677


**SRT Generator Function**

In [5]:
def create_srt(segments, output_file):

    with open(output_file, "w", encoding="utf-8") as f:

        for i, seg in enumerate(segments, start=1):

            start = format_time(seg["start"])
            end = format_time(seg["end"])

            f.write(f"{i}\n")
            f.write(f"{start} --> {end}\n")
            f.write(f"{seg['text']}\n\n")

**Load Whisper Model**

In [6]:
# Old whisper removed. Using Faster-Whisper GPU below.


**Load Summarization Model**

In [7]:
!pip uninstall -y transformers
!pip install transformers==4.52.4 sentencepiece accelerate
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

print("Summarization model loaded")

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Summarization model loaded


In [8]:
from transformers import pipeline

flan_summarizer = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("FLAN-T5 model loaded successfully")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


FLAN-T5 model loaded successfully


In [9]:
def generate_flan_summary(text):

    # limit text size (important for performance)
    text = text[:2000]

    prompt = "Summarize the following text:\n\n" + text

    result = flan_summarizer(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    return result[0]["generated_text"]

**Load Whisper Model (Speech → Text)**

In [10]:
!pip install -q faster-whisper

In [11]:
from faster_whisper import WhisperModel

print("Installed Successfully")

Installed Successfully


In [12]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"

)

print("Large-v3 model loaded")

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Large-v3 model loaded


In [13]:
import torch
print(torch.cuda.is_available())

True


In [14]:
def transcribe_audio(audio_path):

    segments_generator, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
        condition_on_previous_text=False
    )

    transcript = ""
    segments = []

    for seg in segments_generator:

        transcript += seg.text + " "

        segments.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text
        })

    return transcript, segments

In [15]:
# Run process_video() instead of calling transcribe_audio directly.


**Converting Video → Audio**

In [16]:
from moviepy.editor import VideoFileClip
!pip install moviepy==1.0.3


def extract_audio(video_path, audio_path):

    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path)

    print("Audio extracted successfully")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



**Create SRT Generator (Subtitle File)**

In [17]:
def create_srt(segments, output_file):

    def format_time(seconds):
        hrs = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 1000)

        return f"{hrs:02}:{mins:02}:{secs:02},{millis:03}"

    with open(output_file, "w", encoding="utf-8") as f:

        for i, seg in enumerate(segments, start=1):

            start = format_time(seg["start"])
            end = format_time(seg["end"])

            f.write(f"{i}\n")
            f.write(f"{start} --> {end}\n")
            f.write(f"{seg['text']}\n\n")

    print("SRT file created")

**FULL PIPELINE FUNCTION**

In [18]:
def process_video(video_path):

    import os
    os.makedirs("audios", exist_ok=True)
    os.makedirs("transcripts", exist_ok=True)
    os.makedirs("subtitles", exist_ok=True)
    os.makedirs("summaries", exist_ok=True)

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    video_name = video_path.split("/")[-1].split(".")[0]

    audio_path = f"audios/{video_name}.wav"
    transcript_path = f"transcripts/{video_name}.txt"
    srt_path = f"subtitles/{video_name}.srt"
    summary_path = f"summaries/{video_name}_summary.txt"

    # 1. Extract audio
    extract_audio(video_path, audio_path)

    # 2. Whisper transcription
    transcript, segments = transcribe_audio(audio_path)

    # save transcript
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(transcript)

    # 3. SRT file
    create_srt(segments, srt_path)

    # 4. Summaries (BART + FLAN)
    bart_summary = bart_summarizer(transcript[:2000], max_new_tokens=100, min_length=40, do_sample=False)[0]["summary_text"]
    flan_summary = generate_flan_summary(transcript)

    final_summary = f"""BART SUMMARY:\n{bart_summary}\n\nFLAN SUMMARY:\n{flan_summary}"""

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(final_summary)

    print("PROCESS COMPLETED SUCCESSFULLY")

    return transcript, bart_summary, flan_summary

In [19]:
from transformers import pipeline

bart_summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1
)
print("BART loaded on CPU")


Device set to use cpu


BART loaded on CPU


In [20]:
process_video("/content/lecture1.mp4")
process_video("/content/lecture2.mp4")
process_video("/content/lecture3.mp4")

MoviePy - Writing audio in audios/lecture1.wav


MoviePy - Done.
Audio extracted successfully
SRT file created
PROCESS COMPLETED SUCCESSFULLY
MoviePy - Writing audio in audios/lecture2.wav


MoviePy - Done.
Audio extracted successfully
SRT file created
PROCESS COMPLETED SUCCESSFULLY
MoviePy - Writing audio in audios/lecture3.wav


MoviePy - Done.
Audio extracted successfully
SRT file created
PROCESS COMPLETED SUCCESSFULLY


(" The following content is provided under a Creative  Commons license.  Your support will help MIT OpenCourseWare  continue to offer high quality educational resources for free.  To make a donation or to view additional materials  from hundreds of MIT courses, visit MIT OpenCourseWare  at ocw.mit.edu.  JOHN GUTTAGG. Welcome to 6-triple-0-2,  or if you were in 600, the second half of 600.  I'm John Guttag. Let me start with a few administrative things. What's the  workload? There are problem sets. They'll all be programming problems much in the  style of 6.0001 and the goal really twofold. The 6.0001  problem sets were mostly about you learning to be a programmer. A lot of  of that carries over no one learns to be a programmer in half a semester so a lot  of it is to improve your skills but also there's a lot more I would say  conceptual algorithmic material in six triple zero two and the problem sets are  designed to help cement that as well as just to give you programming experience 

In [21]:
!ls audios
!ls transcripts
!ls subtitles
!ls summaries

lecture1.wav  lecture2.wav  lecture3.wav
lecture1.txt  lecture2.txt  lecture3.txt
lecture1.srt  lecture2.srt  lecture3.srt
lecture1_summary.txt  lecture2_summary.txt  lecture3_summary.txt


**Transcript Chunking**

In [22]:
def chunk_transcript(text, chunk_size=1000):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(
            " ".join(words[i:i+chunk_size])
        )

    return chunks

**Read Transcript and Create Chunks**

In [23]:
with open(
    "transcripts/lecture1.txt",
    "r",
    encoding="utf-8"
) as f:

    transcript = f.read()

chunks = chunk_transcript(transcript)

print("Number of chunks:", len(chunks))

Number of chunks: 11


In [24]:
with open(
    "transcripts/lecture2.txt",
    "r",
    encoding="utf-8"
) as f:

    transcript = f.read()

chunks = chunk_transcript(transcript)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [25]:

with open(
    "transcripts/lecture3.txt",
    "r",
    encoding="utf-8"
) as f:

    transcript = f.read()

chunks = chunk_transcript(transcript)

print("Number of chunks:", len(chunks))

Number of chunks: 5


**Section-wise Summaries**

In [26]:
section_summaries = []

for chunk in chunks:

    # Skip empty chunks
    if len(chunk.strip()) < 50:
        continue

    # Limit chunk size for BART
    chunk = chunk[:1000]

    try:
        summary = bart_summarizer(
            chunk,
            max_new_tokens=60,
            min_length=15,
            do_sample=False
        )[0]["summary_text"]

        section_summaries.append(summary)

    except Exception as e:
        print("Skipped chunk:", e)

In [27]:
!ls summaries

lecture1_summary.txt  lecture2_summary.txt  lecture3_summary.txt


In [28]:
!ls subtitles

lecture1.srt  lecture2.srt  lecture3.srt


In [61]:
reference1 = """
The lecture introduces the goals of MIT 6.01 and focuses on engineering reasoning, abstraction, and modularity. Students learn how complex systems are designed, modeled, and analyzed. The lecture also introduces Python programming concepts such as functions, lists, classes, and environments. A practice-theory-practice teaching approach is emphasized throughout the course."""
with open("reference_summary1.txt", "w", encoding="utf-8") as f:
    f.write(reference1)

In [62]:
reference2 = """
The lecture introduces the concepts of signals and systems. Signals are described as functions of independent variables such as time, while systems process and transform signals. Examples including speech and audio signals are discussed to illustrate continuous-time and discrete-time representations. The lecture emphasizes modeling and analyzing signals for engineering applications."""
with open("reference_summary2.txt", "w", encoding="utf-8") as f:
    f.write(reference2)

In [63]:
reference3 = """
The lecture introduces the MIT 6.002 course and discusses the importance of computational problem solving. It reviews prerequisite programming knowledge, object-oriented programming concepts, and Python fundamentals. The instructor explains how computational models can be used to understand and solve real-world problems. The course aims to develop analytical and problem-solving skills through programming."""
with open("reference_summary3.txt", "w", encoding="utf-8") as f:
    f.write(reference3)

In [64]:
with open("reference_lecture1.txt","w",encoding="utf-8") as f:
    f.write("""This lecture introduces the goals and structure of the course. The instructor explains that the course focuses on engineering reasoning, abstraction, modularity, and system design. The course is organized around four modules: software engineering, signals and systems, circuits, and probability and planning.

Students participate in lectures, software labs, design labs, readings, quizzes, and projects. The teaching philosophy is based on "practice, theory, practice," emphasizing learning through hands-on experience.

The lecture introduces important programming concepts in Python, including composition, abstraction, functions, lists, classes, methods, and variables. The instructor explains how complex operations can be built from simpler operations and how data structures can be organized hierarchically.

The lecture also discusses classes, instances, inheritance, and the Python environment model. Students learn how names are associated with values, how functions create local environments, and how Python resolves variable bindings.

The overall goal is to help students develop systematic approaches to designing and understanding complex engineering systems.""")

with open("reference_lecture2.txt","w",encoding="utf-8") as f:
    f.write("""This lecture introduces fundamental concepts of computation and programming. The instructor explains how problems can be represented computationally and solved using algorithms. Students learn about abstraction, modularity, and systematic problem-solving techniques.

The lecture discusses functions, data structures, and program organization. Emphasis is placed on understanding how computational systems process information and how programmers can design reliable and maintainable code.

Several examples demonstrate the relationship between algorithms, efficiency, and correctness. Students are encouraged to think about problem decomposition and software engineering principles when developing solutions.

The lecture also highlights the importance of testing, debugging, and analyzing program behavior. These concepts form the foundation for more advanced topics in computer science and engineering.""")

with open("reference_lecture3.txt","w",encoding="utf-8") as f:
    f.write("""This lecture introduces computational models and their applications in understanding real-world problems. The instructor explains optimization, statistical modeling, and simulation as important tools in data science and engineering.

A major focus of the lecture is the knapsack optimization problem. The instructor discusses objective functions, constraints, brute-force solutions, and greedy algorithms. Students learn how optimization problems can be represented mathematically and solved computationally.

The lecture demonstrates how different greedy strategies can produce different results and explains the distinction between local and global optima. Examples involving food selection and resource allocation are used to illustrate these concepts.

The lecture concludes by emphasizing the importance of algorithm design and computational modeling for making predictions and solving complex real-world problems.""")

In [65]:
import os

files = [
    "reference_lecture1.txt",
    "reference_lecture2.txt",
    "reference_lecture3.txt",
    "reference_summary1.txt",
    "reference_summary2.txt",
    "reference_summary3.txt"
]

for f in files:
    print(f, os.path.exists(f))

reference_lecture1.txt True
reference_lecture2.txt True
reference_lecture3.txt True
reference_summary1.txt True
reference_summary2.txt True
reference_summary3.txt True


In [66]:
!pip install jiwer rouge-score -q

In [69]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = " ".join(text.split())
    return text

In [70]:
reference = clean_text(reference)
generated = clean_text(generated)

In [71]:
!pip install jiwer -q

from jiwer import wer

for i in range(1, 4):

    reference = open(
        f"reference_lecture{i}.txt",
        "r",
        encoding="utf-8"
    ).read()

    generated = open(
        f"transcripts/lecture{i}.txt",
        "r",
        encoding="utf-8"
    ).read()

    score = wer(reference, generated)

    print(f"Lecture {i} WER = {score:.4f}")

Lecture 1 WER = 65.6340
Lecture 2 WER = 31.8468
Lecture 3 WER = 42.6034


In [68]:
from rouge_score import rouge_scorer

for i in range(1, 4):

    reference = open(
        f"reference_summary{i}.txt",
        "r",
        encoding="utf-8"
    ).read()

    generated = open(
        f"summaries/lecture{i}_summary.txt",
        "r",
        encoding="utf-8"
    ).read()

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(reference, generated)

    print(f"\nLecture {i}")
    print("ROUGE-1:", round(scores['rouge1'].fmeasure, 4))
    print("ROUGE-2:", round(scores['rouge2'].fmeasure, 4))
    print("ROUGE-L:", round(scores['rougeL'].fmeasure, 4))


Lecture 1
ROUGE-1: 0.233
ROUGE-2: 0.0198
ROUGE-L: 0.1359

Lecture 2
ROUGE-1: 0.2135
ROUGE-2: 0.0341
ROUGE-L: 0.1573

Lecture 3
ROUGE-1: 0.2286
ROUGE-2: 0.0583
ROUGE-L: 0.1905


In [43]:
import os

print(os.listdir("summaries"))

['lecture2_summary.txt', 'lecture3_summary.txt', 'lecture1_summary.txt']


In [44]:
print(open("reference_summary1.txt","r",encoding="utf-8").read())

print("\n" + "="*50 + "\n")

print(open("summaries/lecture1_summary.txt","r",encoding="utf-8").read())


The lecture introduces the course objectives, structure, and teaching methodology. It emphasizes abstraction, modularity, and engineering reasoning as key concepts. Students learn about Python fundamentals including functions, lists, classes, and variable binding. The course uses a practice-oriented approach through lectures, labs, and projects. The goal is to build skills for designing and understanding complex systems.



BART SUMMARY:
Danny Freeman: 6.01 is mostly about modes of reasoning. We want to talk about how do you design and build complicated systems? That's what engineers do. We're very good at it, and you know that from your common  everyday experience.

FLAN SUMMARY:
Donate to MIT OpenCourseWare.
